# AI GeoImage Coreg - Default Colab Run

This notebook runs the **default coregistration pipeline** from this repository in **Google Colab**.

**Recommended runtime:** Google Colab with **GPU (T4)**.

**Where files are stored:** inputs and outputs in this workflow are stored on **Google Drive** (mounted to `/content/drive`).

[![Run in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atsyplenkov/ai_geoimage_coreg/blob/main/examples/default_coreg_colab.ipynb)

## 1) Verify GPU Runtime

What this does: confirms that Colab sees a CUDA GPU before running heavy matching.

What you need to do: if this cell fails, switch runtime to **Runtime -> Change runtime type -> T4 GPU** and rerun.

In [ ]:
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('GPU is not enabled. In Colab: Runtime -> Change runtime type -> T4 GPU.')

## 2) Mount Google Drive

What this does: mounts your Drive at `/content/drive` so notebook cells can read/write files in `MyDrive`.

What you need to do: approve the Drive authorization prompt.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3) Set Input and Output Paths on Drive

What this does: defines where your historical image, modern reference image, and outputs live on Drive.

What you need to do: edit these path variables if your files are in a different Drive folder.

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ai_geoimage_coreg')
HISTORICAL_IMAGE = DRIVE_ROOT / 'historical.tif'
REFERENCE_IMAGE = DRIVE_ROOT / 'modern.tif'
OUTPUT_PREFIX = DRIVE_ROOT / 'georef'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'DRIVE_ROOT: {DRIVE_ROOT}')
print('If your files are elsewhere in Drive, edit DRIVE_ROOT/HISTORICAL_IMAGE/REFERENCE_IMAGE above.')

### Required Input Files

Upload your two input GeoTIFF files to `DRIVE_ROOT` (or update the paths above):

- Historical image: `historical.tif`
- Georeferenced modern reference image: `modern.tif`

Both files are expected to be on **Google Drive**.

## 4) Install Dependencies in Colab

What this does: installs GDAL system packages and the latest repository code from GitHub.

What you need to do: wait for install to finish; if imports fail afterward, restart the runtime and rerun from the top.

In [ ]:
!apt-get update
!apt-get install -y gdal-bin libgdal-dev
!pip install --upgrade pip
!pip install git+https://github.com/atsyplenkov/ai_geoimage_coreg.git

## 5) Validate Imports

What this does: checks that GDAL, Rasterio, and `run_pipeline` import correctly before processing.

In [ ]:
from osgeo import gdal
import rasterio
from ai_geoimage_coreg.core import run_pipeline

print('GDAL version:', gdal.VersionInfo())
print('Rasterio version:', rasterio.__version__)
print('Imports OK. If these imports fail after install, restart runtime and rerun cells.')

## 6) Confirm Input Files Exist

What this does: stops early with clear paths if required input TIFFs are missing.

What you need to do: upload missing files to Drive or edit the path variables.

In [ ]:
missing = [str(p) for p in [HISTORICAL_IMAGE, REFERENCE_IMAGE] if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Missing input files:\n'
        + '\n'.join(missing)
        + '\n\nUpload both TIFFs to DRIVE_ROOT or edit the path variables above.'
    )

print('Input files found.')

## 7) Run Default Coregistration

What this does: runs the package default pipeline (`run_pipeline`) and writes outputs to Google Drive.

In [ ]:
run_pipeline(
    path_hex=str(HISTORICAL_IMAGE),
    path_ref=str(REFERENCE_IMAGE),
    output_prefix=str(OUTPUT_PREFIX),
)

## 8) Check Output Files

What this does: verifies expected result files are present in your Drive output folder.

In [ ]:
expected_outputs = [
    DRIVE_ROOT / 'georef_raw.csv',
    DRIVE_ROOT / 'georef_clean.csv',
    DRIVE_ROOT / 'georef_poly.tif',
    DRIVE_ROOT / 'georef_tps.tif',
]

for out in expected_outputs:
    print(f'{out.name}: {"OK" if out.exists() else "MISSING"}')